# Vector-Only Retrieval Evaluation

This notebook runs a **vector-only** retrieval evaluation against the frozen QA benchmark (`data/qa_final.jsonl`). It uses [`VectorRetriever`](../src/retrieval/retriever.py) backed by a **local FAISS index + SQLite payload cache** (via [`SQLitePayloadFaissVectorStore`](../src/retrieval/sqlite_faiss_store.py) / [`build_vector_retriever`](../src/evaluation/retriever_factory.py)) — **Qdrant is not used**.

No knowledge-graph, `GraphExpansion`, `GraphTraversal`, or hybrid-fusion objects appear anywhere in this notebook, and it does not import from or read `notebooks/archive/`.

See [`specs/008-vector-retrieval-eval-notebook/quickstart.md`](../specs/008-vector-retrieval-eval-notebook/quickstart.md) for the operator guide and validation scenarios A-E.

## 1. Environment setup

In [ ]:
# Optional: install runtime dependencies if your environment does not have them yet.
# Uncomment and run once if needed.
# %pip install -q faiss-cpu sentence-transformers pandas


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    # Useful if the notebook is launched from notebooks/ (path portability).
    PROJECT_ROOT = Path.cwd().parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")


## 2. Configuration

Edit only this cell to change run parameters (FR-006). No `src/` edits are required.

In [ ]:
from datetime import datetime, timezone

# --- Benchmark input ---
QA_PATH = PROJECT_ROOT / "data/qa_final.jsonl"

# --- Retrieval parameters ---
TOP_K_LIST = [1, 5, 10]
INDEX_DIR = PROJECT_ROOT / "data/faiss_index"
MODEL_NAME = "intfloat/multilingual-e5-large"
SCORE_THRESHOLD = 0.3
EXPAND_UNITS = True

# --- Run controls ---
SAMPLE_LIMIT = None  # e.g. 25 for a fast smoke test; None evaluates all eligible cases
DEV_HASHING = False  # True routes to an in-memory hashing embedder/store (no FAISS index/model needed)

# --- Output (only used if the persistence cell is triggered) ---
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT_DIR = PROJECT_ROOT / "evaluation_runs" / "vector_only" / RUN_ID

print("Vector-only retrieval evaluation — configuration")
print(f"  QA_PATH         = {QA_PATH}")
print(f"  TOP_K_LIST      = {TOP_K_LIST}")
print(f"  INDEX_DIR       = {INDEX_DIR}")
print(f"  MODEL_NAME      = {MODEL_NAME}")
print(f"  SCORE_THRESHOLD = {SCORE_THRESHOLD}")
print(f"  EXPAND_UNITS    = {EXPAND_UNITS}")
print(f"  SAMPLE_LIMIT    = {SAMPLE_LIMIT}")
print(f"  DEV_HASHING     = {DEV_HASHING}")
print(f"  OUT_DIR         = {OUT_DIR}")


## 3. Load the QA benchmark (FR-011: clear error if missing)

In [ ]:
from evaluation.io_utils import read_jsonl

if not QA_PATH.exists():
    raise FileNotFoundError(
        f"qa_final.jsonl not found at {QA_PATH}. Set QA_PATH in the configuration cell to the correct location."
    )

qa_rows = list(read_jsonl(QA_PATH))
print(f"Loaded {len(qa_rows)} raw QA rows from {QA_PATH}")


## 4. Eligibility filtering (FR-003, FR-007)

Every row is classified into exactly one of: eligible / skipped (unanswerable) / skipped (missing ground truth).

In [ ]:
from evaluation.eligibility import select_eligible_cases

eligibility_summary = select_eligible_cases(qa_rows, sample_limit=SAMPLE_LIMIT)

print("Eligibility summary")
print(f"  total_rows                   = {eligibility_summary.total_rows}")
print(f"  eligible (evaluated)          = {len(eligibility_summary.eligible)}")
print(f"  skipped_unanswerable          = {eligibility_summary.skipped_unanswerable}")
print(f"  skipped_missing_ground_truth  = {eligibility_summary.skipped_missing_ground_truth}")

if SAMPLE_LIMIT is not None:
    print(
        f"\nNote: SAMPLE_LIMIT={SAMPLE_LIMIT} was applied — evaluated "
        f"{len(eligibility_summary.eligible)} eligible case(s) capped at SAMPLE_LIMIT "
        f"(rows beyond the cap were not examined)."
    )


## 5. Build the vector retriever (FAISS + SQLite payload cache; FR-002a, FR-012)

**Qdrant is not used by this notebook.** `DEV_HASHING=True` routes to an in-memory hashing embedder/store for the smoke-test scenario; otherwise a real FAISS index directory (`INDEX_DIR`) is required.

In [ ]:
from evaluation.retriever_factory import RetrieverRuntimeConfig, build_vector_retriever

max_k = max(TOP_K_LIST)
runtime_config = RetrieverRuntimeConfig(
    store="faiss",
    index_dir=INDEX_DIR,
    model=MODEL_NAME,
    dev_hashing=DEV_HASHING,
    top_k=max(max_k * 3, 30),
    top_n=max_k,
    score_threshold=SCORE_THRESHOLD,
    expand_units=EXPAND_UNITS,
)

try:
    retriever = build_vector_retriever(runtime_config)
except FileNotFoundError as exc:
    raise FileNotFoundError(
        f"Could not build the FAISS-backed retriever: {exc}\n"
        "Either set DEV_HASHING = True for a smoke test, or point INDEX_DIR at a directory "
        "containing 'index.faiss' and 'payloads.jsonl' (see scripts/build_vector_index.py)."
    ) from exc

print(f"Retriever ready (store={'dev_hashing' if DEV_HASHING else runtime_config.store!r}).")


## 6. Retrieval + scoring loop (FR-004, FR-014)

`cases` is freshly re-initialized every time this cell runs, so reruns after a configuration change never mix stale per-case rows into the new aggregate.

In [ ]:
from evaluation.retrieval_eval_report import RetrievalCaseResult, build_case_metrics_row, metric_keys_for

metric_keys = metric_keys_for(TOP_K_LIST)
cases: list[RetrievalCaseResult] = []  # re-initialized on every execution (FR-014)
error_count = 0

for case in eligibility_summary.eligible:
    try:
        result = retriever.retrieve(case.question, filter_profile="broad", top_n=max_k)
        retrieved_chunk_ids = [chunk.chunk_id for chunk in result.chunks]
        metrics_row = build_case_metrics_row(retrieved_chunk_ids, case.ground_truth_chunk_ids, TOP_K_LIST)
        cases.append(
            RetrievalCaseResult(
                qa_id=case.qa_id,
                mode="vector_only",
                question=case.question,
                category=case.category,
                difficulty=case.difficulty,
                answer_type=case.answer_type,
                ground_truth_chunk_ids=sorted(case.ground_truth_chunk_ids),
                retrieved_chunk_ids=retrieved_chunk_ids,
                metrics=metrics_row,
            )
        )
    except Exception as exc:  # surfaced per-case, does not abort the whole run
        error_count += 1
        cases.append(
            RetrievalCaseResult(
                qa_id=case.qa_id,
                mode="vector_only",
                question=case.question,
                category=case.category,
                difficulty=case.difficulty,
                answer_type=case.answer_type,
                ground_truth_chunk_ids=sorted(case.ground_truth_chunk_ids),
                retrieved_chunk_ids=[],
                metrics={key: 0.0 for key in metric_keys},
                error=str(exc),
            )
        )

print(f"Evaluated {len(cases)} case(s); error_count={error_count}")


## 7. Overall + breakdown metrics tables (FR-005)

In [ ]:
import pandas as pd

from evaluation.metrics import aggregate, aggregate_by

case_dicts = [{"category": c.category, "difficulty": c.difficulty, "answer_type": c.answer_type, **c.metrics} for c in cases]

overall = aggregate(case_dicts, metric_keys)
by_category = aggregate_by(case_dicts, "category", metric_keys)
by_difficulty = aggregate_by(case_dicts, "difficulty", metric_keys)
by_answer_type = aggregate_by(case_dicts, "answer_type", metric_keys)

print("=== VECTOR-ONLY retrieval evaluation — run summary ===")
print(f"total_rows                  = {eligibility_summary.total_rows}")
print(f"evaluated                   = {len(cases)}")
print(f"skipped_unanswerable         = {eligibility_summary.skipped_unanswerable}")
print(f"skipped_missing_ground_truth = {eligibility_summary.skipped_missing_ground_truth}")
print(f"error_count                  = {error_count}")

print("\nOverall metrics:")
display(pd.DataFrame([overall]))

print("\nBy category:")
display(pd.DataFrame.from_dict(by_category, orient="index"))

print("\nBy difficulty:")
display(pd.DataFrame.from_dict(by_difficulty, orient="index"))

print("\nBy answer_type:")
display(pd.DataFrame.from_dict(by_answer_type, orient="index"))


## 8. Per-case results (Acceptance Scenario 2, US1)

In [ ]:
per_case_rows = [
    {
        "qa_id": c.qa_id,
        "question": c.question,
        "ground_truth_chunk_ids": c.ground_truth_chunk_ids,
        "retrieved_chunk_ids": c.retrieved_chunk_ids,
        "error": c.error,
        **c.metrics,
    }
    for c in cases
]
per_case_df = pd.DataFrame(per_case_rows)
display(per_case_df)


## 9. Optional: persist artifacts (FR-008)

Writes `retrieval_cases.jsonl` and `retrieval_metrics.json` under a fresh `OUT_DIR` (`evaluation_runs/vector_only/<run_id>/`). Skipped by default — set `PERSIST = True` below and re-run this cell to trigger it.

In [ ]:
from evaluation.retrieval_eval_report import ModeRunSummary, write_case_jsonl, write_metrics_json

PERSIST = False  # flip to True and re-run this cell to persist artifacts

if PERSIST:
    run_summary = ModeRunSummary(
        mode="vector_only",
        config={
            "index_dir": str(INDEX_DIR),
            "model": MODEL_NAME,
            "score_threshold": SCORE_THRESHOLD,
            "expand_units": EXPAND_UNITS,
            "top_k": TOP_K_LIST,
            "sample_limit": SAMPLE_LIMIT,
            "dev_hashing": DEV_HASHING,
        },
        total_rows=eligibility_summary.total_rows,
        evaluated=len(cases),
        skipped_unanswerable=eligibility_summary.skipped_unanswerable,
        skipped_missing_ground_truth=eligibility_summary.skipped_missing_ground_truth,
        error_count=error_count,
        overall=overall,
        by_category=by_category,
        by_difficulty=by_difficulty,
        by_answer_type=by_answer_type,
    )
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    write_case_jsonl(OUT_DIR / "retrieval_cases.jsonl", cases)
    write_metrics_json(OUT_DIR / "retrieval_metrics.json", run_summary, run_summary.config)
    print(f"Persisted artifacts under {OUT_DIR}")
else:
    print("PERSIST is False — nothing written. Set PERSIST = True above and re-run this cell to persist.")
